In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

/home/foolmann/miniconda3/envs/genaienv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os


exchange_api_key = os.getenv("CURRENCY_EXCHANGE_API_KEY")

# print(exchange_api_key)

In [89]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [93]:
# tool creation 

@tool 
def get_conversion_factor(base_currency: str,target_currency:str ) -> float:
    """ This function fetches the currency conversion factor between a given base currency and a target currency"""
    url = f"https://v6.exchangerate-api.com/v6/{exchange_api_key}/latest/{base_currency}"

    response = requests.get(url)
    # print(response.conversion_rates)
    response = response.json()
    response = response['conversion_rates'][target_currency]

    return response

@tool 
def convert(base_currency_value:int, conversion_rate:Annotated[float,InjectedToolArg ]) -> float:
    """given a currency conversion rate this function calculates the target currency value from a given base currency value"""
    return base_currency_value*conversion_rate


# This annotate and injectedtoolarg . Tells llm that do not fill the values I (developer/runtime) will inject this value after running earlier tools 

# # MODIFIED TO AVOID THE TWO MESSAGE TO TOOL CALLS
# @tool
# def convert(amount:int, from_currency:str, to_currency:str) -> float:
#     """Convert one currency to another."""
#     rate = get_conversion_factor.invoke({'base_currency':from_currency,'target_currency':to_currency})
#     return amount * rate

In [91]:
rate_returned = get_conversion_factor.invoke({'base_currency':'NPR','target_currency':'USD'})
rate_returned

0.006527

In [95]:
convert.invoke({'base_currency_value':100,'conversion_rate':rate_returned})

0.6527

Tool binding 

In [96]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.2)

In [119]:
llm_with_tools =llm.bind_tools([get_conversion_factor,convert])

In [113]:
messages=[]

In [114]:
messages = [HumanMessage("Convert 10 USD to NPR ")]

In [115]:
messages

[HumanMessage(content='Convert 10 USD to NPR ', additional_kwargs={}, response_metadata={})]

In [116]:
ai_message = llm_with_tools.invoke(messages)

In [120]:
ai_message

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "NPR"}'}, '__gemini_function_call_thought_signatures__': {'051192cb-73a9-4410-9e44-0784b2cb541d': 'EjQKMgERTTIP+akWxsZNThw+SjB4d/eM4p6fOBNKDLR2JjvCozawOVlBnDEjFgTDFtuYDqm3'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fb330-dfc4-7871-a2c1-0a74a995449d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': '051192cb-73a9-4410-9e44-0784b2cb541d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 29, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}})

In [121]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': '051192cb-73a9-4410-9e44-0784b2cb541d',
  'type': 'tool_call'}]

This will not call the both tools at once. This is because llm doesnot know the rates yet so convert function tool is not called. To avoid this we have to modify the convert with the tool calling inside it so that llm understands for the conversion both tool are needed. Other wise we have to first convert rate gathering then with another invoke we have to do conversion 

In [122]:
ai_message.tool_calls[0]

{'name': 'get_conversion_factor',
 'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
 'id': '051192cb-73a9-4410-9e44-0784b2cb541d',
 'type': 'tool_call'}

In [123]:
tool_msg = get_conversion_factor.invoke(ai_message.tool_calls[0])

In [124]:
messages.append(tool_msg)
messages

[HumanMessage(content='Convert 10 USD to NPR ', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='153.2195', name='get_conversion_factor', tool_call_id='051192cb-73a9-4410-9e44-0784b2cb541d')]

In [ ]:
final_msg = llm_with_tools.invoke(messages)
final_msg

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "NPR", "base_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'9c6dd036-57c2-4512-b052-95fc4aaef9d7': 'EjQKMgERTTIPIiqK/VV9uhLvQ9v+3x0IZQ5dCYmhdgymGI77zh+novyXMLruNg0Ae9pmiZUm'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fb336-846f-7232-8ace-abc8b358a77d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'NPR', 'base_currency': 'USD'}, 'id': '9c6dd036-57c2-4512-b052-95fc4aaef9d7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 29, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}})

This is so unreliable so better is to use the single tool with the nested function calling

In [135]:

def get_conversion_factor(base_currency: str,target_currency:str ) -> float:
    """ This function fetches the currency conversion factor between a given base currency and a target currency"""
    url = f"https://v6.exchangerate-api.com/v6/{exchange_api_key}/latest/{base_currency}"

    response = requests.get(url)
    # print(response.conversion_rates)
    response = response.json()
    response = response['conversion_rates'][target_currency]

    return response




# MODIFIED TO AVOID THE TWO MESSAGE TO TOOL CALLS
@tool
def convert(amount:int, from_currency:str, to_currency:str) -> float:
    """Convert one currency to another."""
    rate = get_conversion_factor(from_currency,to_currency)
    return amount * rate

In [136]:
llm_with_tools =llm.bind_tools([convert])

In [137]:
messages=[]

In [138]:
messages = [HumanMessage("Convert 10 USD to NPR ")]

In [139]:
res = llm_with_tools.invoke(messages)

In [140]:
res

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"to_currency": "NPR", "amount": 10, "from_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'1a768048-95f6-4f49-b41a-96c5332f6280': 'EjQKMgERTTIP5lQ126mdaPhkq9m/L0DYr0DfzlIqPUB9k2ItQEg+ZqilsRfokK2Xjjynxsqb'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fb33d-e304-7f91-bf18-3e148e980b95-0', tool_calls=[{'name': 'convert', 'args': {'to_currency': 'NPR', 'amount': 10, 'from_currency': 'USD'}, 'id': '1a768048-95f6-4f49-b41a-96c5332f6280', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 30, 'total_tokens': 112, 'input_token_details': {'cache_read': 0}})

In [141]:
res.tool_calls

[{'name': 'convert',
  'args': {'to_currency': 'NPR', 'amount': 10, 'from_currency': 'USD'},
  'id': '1a768048-95f6-4f49-b41a-96c5332f6280',
  'type': 'tool_call'}]

In [142]:
tool_msg = convert.invoke(res.tool_calls[0])

In [143]:
tool_msg

ToolMessage(content='1532.1950000000002', name='convert', tool_call_id='1a768048-95f6-4f49-b41a-96c5332f6280')